# 01 — Understand the observe–plan–act–verify loop

You will inspect a five-tool runtime and see how a missing unit stops an analysis. This is a deterministic workflow baseline, not an external AI model. Everything is synthetic. No API key or GPU is needed. The following cell loads code from your local clone or a pinned public source; inspect that source before executing it.

In [ ]:
from pathlib import Path
import hashlib, importlib.util, json, sys, urllib.request

# Replaced during publication with the commit containing the reviewed helper.
REF = "faf63f0f09cba07291e8c3d09ef95974b8404d77"
candidates = [Path("agent_studio.py"), Path("../agent_studio.py")]
source_path = next((p.resolve() for p in candidates if p.is_file()), None)
origin = "local copy"
if source_path is None:
    origin = f"GitHub commit {REF}"
    url = ("https://raw.githubusercontent.com/ZagElSayed/"
           f"Introduction-to-Neuroinformatics/{REF}/"
           "chapters/ch19-ai-agents/agent_studio.py")
    payload = urllib.request.urlopen(url, timeout=30).read()
    source_path = Path("agent_studio.py").resolve()
    source_path.write_bytes(payload)
manifest = {"origin": origin, "source_path": str(source_path),
            "sha256": hashlib.sha256(source_path.read_bytes()).hexdigest()}
Path("agent_source_manifest.json").write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest, indent=2))
spec = importlib.util.spec_from_file_location("agent_studio", source_path)
studio_module = importlib.util.module_from_spec(spec)
sys.modules["agent_studio"] = studio_module
spec.loader.exec_module(studio_module)
Studio = studio_module.Studio

## Start with a bounded request

The request identifies one dataset. The runtime chooses from registered tools and writes only teaching outputs under the directory you select. Inspect the proposal schema and tool names first.

In [ ]:
print(json.dumps(studio_module.TOOL_SCHEMA, indent=2))
studio = Studio("notebook01_outputs")
run = studio.run("toy_good")
print(run["status"], run["reason"])
for event in run["trajectory"]:
    print(event["step"], event.get("proposal"), event["status"])

## Inspect evidence, not just the final status

Compare the source identifier, recipe, QC scope, and computed alpha power. Open the saved report; the source signal has declared illustrative units, not participant physiology.

In [ ]:
print(json.dumps(run["values"], indent=2))
receipt = run["values"].get("stage_report")
if receipt:
    print(Path(receipt["path"]).read_text())
print("Trajectory log:", run["log"]["path"])

## Observe a correct stop

The next fixture lacks units. No unit is guessed, and the spectral tool should remain uncalled. Compare the observed state with that intended behavior. Then try `toy_flat`, `toy_nonfinite`, and `toy_wrong_rate`.

In [ ]:
stopped = studio.run("toy_missing_units")
print(stopped["status"], stopped["reason"])
print(stopped["values"].get("read_qc"))
print("Computed spectrum:", "compute_alpha" in stopped["values"])

## Explain what happened

Write a paragraph identifying observations, choices, effects, verification, and stop conditions. Which actions were pure reads? Which created files? Why is a correct refusal different from a failed scientific computation?